In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [7]:
df = pd.read_csv("car.csv")


In [8]:
df['Car_Age'] = 2025 - df['Year']  # Calculate age of the car
df.drop(['Car_Name', 'Year'], axis=1, inplace=True)

In [9]:
X = df.drop('Selling_Price', axis=1)
y = df['Selling_Price']

In [10]:
categorical_cols = ['Fuel_Type', 'Seller_Type', 'Transmission']

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first'), categorical_cols)
    ],
    remainder='passthrough'
)

In [13]:
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])


In [14]:
# Random Forest pipeline
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

In [15]:
lr_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)

/usr/local/lib/python3.11/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(drop='first'),
                                                  ['Fuel_Type', 'Seller_Type',
                                                   'Transmission'])])),
                ('regressor', RandomForestRegressor(random_state=42))])

In [16]:
lr_preds = lr_pipeline.predict(X_test)
rf_preds = rf_pipeline.predict(X_test)

In [17]:
def evaluate_model(name, y_true, y_pred):
    print(f"--- {name} ---")
    print("R² Score:", r2_score(y_true, y_pred))
    print("MAE:", mean_absolute_error(y_true, y_pred))
    print("RMSE:", np.sqrt(mean_squared_error(y_true, y_pred)))
    print()

In [18]:
evaluate_model("Linear Regression", y_test, lr_preds)
evaluate_model("Random Forest", y_test, rf_preds)

--- Linear Regression ---
R² Score: 0.8489813024897863
MAE: 1.2162256821301358
RMSE: 1.8651552135521248

--- Random Forest ---
R² Score: 0.9599365174684459
MAE: 0.6389540983606562
RMSE: 0.960669421826669



In [19]:
import joblib

# Save the Random Forest model pipeline
joblib.dump(rf_pipeline, 'car_price_model.pkl')
print("✅ Model saved as 'car_price_model.pkl'")


✅ Model saved as 'car_price_model.pkl'


In [20]:
# Load the model
model = joblib.load('car_price_model.pkl')

# Example input (same format as original features)
sample_input = pd.DataFrame([{
    'Present_Price': 9.5,
    'Kms_Driven': 40000,
    'Fuel_Type': 'Petrol',
    'Seller_Type': 'Dealer',
    'Transmission': 'Manual',
    'Owner': 0,
    'Car_Age': 5
}])

# Make prediction
predicted_price = model.predict(sample_input)
print(" Predicted Selling Price:", round(predicted_price[0], 2), "Lakhs")


 Predicted Selling Price: 8.02 Lakhs
